In [ ]:
from spatial_manifolds.cross_correlograms import compute_CCH, classify_CCH
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
import pynapple as nap
from spatial_manifolds.toroidal import *
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.data.binning import get_bin_config
from spatial_manifolds.mlencoding import *
from spatial_manifolds.circular_decoder import circular_decoder, cross_validate_decoder, cross_validate_decoder_time, circular_nanmean
from spatial_manifolds.data.curation import curate_clusters
from scipy.stats import zscore
from spatial_manifolds.util import gaussian_filter_nan
from spatial_manifolds.predictive_grid import compute_travel_projected, wrap_list
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *
import xarray as xr
from scipy.signal import convolve

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

fig_path = '/Users/harryclark/Documents/figs/FIGURE1/'


In [ ]:
# load dataframes and find high pR2 connected pairs
import os
import glob

data_dir = '/Users/harryclark/Downloads/COHORT12/xgboost_task_anchoring'
all_files = glob.glob(os.path.join(data_dir, "*.csv"))

data = pd.DataFrame()
for file in all_files:
    df = pd.read_csv(file)

    if 'X_pos_cov' not in df.columns or 'Y_pos_cov' not in df.columns:
        print(f"Skipping file {file} due to missing columns.")
        continue

    data = pd.concat([data, df], ignore_index=True)

print(f"Loaded {len(all_files)} files. Combined data shape: {data.shape}")

# Calculate difference in X and Y between covariate and test positions
data['delta_X'] = data['X_pos_cov'] - data['X_pos_test']
data['delta_Y'] = data['Y_pos_cov'] - data['Y_pos_test']

# Create unique id columns for cluster_id and cluster_id_cov
data['unique_cluster_id'] = (
    data['mouse'].astype(str) + '_' +
    data['day'].astype(str) + '_' +
    data['cluster_id'].astype(str)
)
data['unique_cluster_id_cov'] = (
    data['mouse'].astype(str) + '_' +
    data['day'].astype(str) + '_' +
    data['cluster_id_cov'].astype(str)
)

# Subset to only firing_mode == 'session'
data_session = data[data['firing_mode'] == 'session']

# DataFrame where position_in_covariate is True and n_neurons is 0
position_only = data_session[(data_session['position_in_covariate'] == True) & (data_session['n_neurons'] == 0)]

# DataFrame where position_in_covariate is True and n_neurons is 1
position_single = data_session[(data_session['position_in_covariate'] == True) & (data_session['n_neurons'] == 1)]

neuron_only = data_session[(data_session['position_in_covariate'] == False) & (data_session['n_neurons'] == 1)]

In [ ]:
top10_pr2cv = neuron_only.nlargest(10, 'pR2_cv')
top10_pr2cv

In [ ]:
source_path = '/Users/harryclark/Downloads/COHORT12/'


In [ ]:
results = []

for (mouse, day), group in neuron_only.groupby(['mouse', 'day']):
    print(f"Processing mouse {mouse}, day {day}")
    cluster_ids = group['cluster_id'].unique()
    
    # Load session data
    vr_folder = f'{source_path}M{mouse}/D{day:02}/VR/'
    spikes_path = vr_folder + f"sub-{mouse}_day-{day:02}_ses-VR_srt-kilosort4_clusters.npz"
    clusters = nap.load_file(spikes_path)
    clusters = curate_clusters(clusters)
    
    # Compute CCH for all pairs
    CCH = compute_CCH(clusters, list(cluster_ids))
    
    # Compute baseline
    bin_size_sec = 0.001
    std_sec = 0.005
    support = 3 * std_sec
    x = np.arange(-support, support + bin_size_sec, bin_size_sec)
    gaussian = np.exp(-0.5 * (x / std_sec) ** 2)
    gaussian[x == 0] *= 0.6
    gaussian /= gaussian.sum()
    CCH_baseline = xr.apply_ufunc(
        lambda x: convolve(x, gaussian, mode="same"),
        CCH,
        input_core_dims=[["lag"]],
        output_core_dims=[["lag"]],
        vectorize=True,
        output_dtypes=[float],
    )
    CCH_baseline.name = "cch_baseline"
    
    # Classify connections
    fc = xr.apply_ufunc(
        classify_CCH,
        CCH,
        CCH_baseline,
        input_core_dims=[["lag"], ["lag"]],
        vectorize=True,
        kwargs={"lags": CCH.coords["lag"].values, "bin_size_sec": bin_size_sec},
        output_dtypes=[bool],
    )
    fc = fc.to_series().reset_index()
    fc = fc.rename({"source": "from", "target": "to", 0: "connected"}, axis=1)
    fc['mouse'] = mouse
    fc['day'] = day

    fc['unique_cluster_id'] = (
    fc['mouse'].astype(str) + '_' +
    fc['day'].astype(str) + '_' +
    fc['from'].astype(str)
    )
    fc['unique_cluster_id_cov'] = (
    fc['mouse'].astype(str) + '_' +
    fc['day'].astype(str) + '_' +
    fc['to'].astype(str)
    )

    results.append(fc)

# Combine all results
final_df = pd.concat(results, ignore_index=True)

In [ ]:
final_df

In [ ]:
neuron_only

In [ ]:
merged = pd.merge(
    final_df,
    neuron_only[['unique_cluster_id', 'unique_cluster_id_cov', 'pR2_cv']],
    on=['unique_cluster_id', 'unique_cluster_id_cov'],
    how='left'
)

In [ ]:
import matplotlib.pyplot as plt

# Group by connection status and compute mean pR2_cv
mean_pr2 = merged.groupby('connected')['pR2_cv'].mean()

# Bar plot
plt.figure(figsize=(4, 5))
plt.bar(['Connected', 'Unconnected'], mean_pr2.values, color=['#6897CD', '#c04744'])
plt.ylabel('Mean pR2_cv')
plt.title('Mean pR2_cv: Connected vs Unconnected')
plt.tight_layout()

In [ ]:
conn_status

In [ ]:
import matplotlib.pyplot as plt

# Merge connection info with neuron_only for both 'from' and 'to' cells
conn_from = connected_cells[['mouse', 'day', 'from', 'to']].rename(columns={'from': 'cluster_id'})
conn_from['connected'] = True
unconn_from = unconnected_cells[['mouse', 'day', 'from', 'to']].rename(columns={'from': 'cluster_id'})
unconn_from['connected'] = False

conn_status = pd.concat([conn_from, unconn_from], ignore_index=True)

# Merge with neuron_only to get pR2_cv values for each cell
merged = pd.merge(
    neuron_only,
    conn_status,
    on=['mouse', 'day', 'cluster_id'],
    how='inner'
)


In [ ]:
import matplotlib.pyplot as plt

# Merge connection info with neuron_only for both 'from' and 'to' cells
conn_from = connected_cells[['mouse', 'day', 'from', 'to']].rename(columns={'from': 'cluster_id'})
conn_from['connected'] = True
unconn_from = unconnected_cells[['mouse', 'day', 'from', 'to']].rename(columns={'from': 'cluster_id'})
unconn_from['connected'] = False

conn_status = pd.concat([conn_from, unconn_from], ignore_index=True)

# Merge with neuron_only to get pR2_cv values for each cell
merged = pd.merge(
    neuron_only,
    conn_status,
    on=['mouse', 'day', 'cluster_id'],
    how='inner'
)

# Group by connection status and compute mean pR2_cv
mean_pr2 = merged.groupby('connected')['pR2_cv'].mean()

# Bar plot
plt.figure(figsize=(4, 5))
plt.bar(['Connected', 'Unconnected'], mean_pr2.values, color=['#6897CD', '#c04744'])
plt.ylabel('Mean pR2_cv')
plt.title('Mean pR2_cv: Connected vs Unconnected')
plt.tight_layout()
plt.show()

In [ ]:
cluster_ids = neuron_only[neuron_only['mouse'] == mouse][neuron_only['day'] == day]['cluster_id'].unique()
mouse = 29
day =23

# Load data
vr_folder = f'{source_path}M{mouse}/D{day:02}/VR/'
spikes_path = vr_folder + f"sub-{mouse}_day-{day:02}_ses-VR_srt-kilosort4_clusters.npz"
beh_path = vr_folder + f"sub-{mouse}_day-{day:02}_ses-VR_beh.nwb"
beh = nap.load_file(beh_path)
clusters = nap.load_file(spikes_path)
#print(f'there are this many clusters before curation {len(clusters)}')
clusters = curate_clusters(clusters)

# Compute CCH
CCH = compute_CCH(clusters, list(cluster_ids))

# Compute baseline CCH
bin_size_sec = 0.001
std_sec = 0.005
support = 3 * std_sec
support_bins = int(support / bin_size_sec)
x = np.arange(-support, support + bin_size_sec, bin_size_sec)
gaussian = np.exp(-0.5 * (x / std_sec) ** 2)
gaussian[x == 0] *= 0.6
gaussian /= gaussian.sum()
CCH_baseline = xr.apply_ufunc(
    lambda x: convolve(x, gaussian, mode="same"),
    CCH,
    input_core_dims=[["lag"]],
    output_core_dims=[["lag"]],
    vectorize=True,
    output_dtypes=[float],
)
CCH_baseline.name = "cch_baseline"

# Classify
fc = xr.apply_ufunc(
    classify_CCH,
    CCH,
    CCH_baseline,
    input_core_dims=[["lag"], ["lag"]],
    vectorize=True,
    kwargs={"lags": CCH.coords["lag"].values, "bin_size_sec": 0.001},
    output_dtypes=[bool],
)
fc = fc.to_series().reset_index()
fc = fc.rename({"source": "from", "target": "to", 0: "connected"}, axis=1)

import matplotlib.pyplot as plt

connected = fc[fc["connected"]]
lag_arrays = [
    CCH.sel(source=fr, target=to)
    for fr, to in zip(connected["from"], connected["to"])
]

from scipy.ndimage import gaussian_filter1d

for i in range(len(lag_arrays)):
    plt.bar(
        x=CCH.coords["lag"].values,
        height=gaussian_filter1d(lag_arrays[i].values, sigma=1.0),
        width=0.001,
    )
    plt.show()


In [ ]:
for cell_target, cell_covariate in zip(top10_pr2cv["unique_cluster_id"], top10_pr2cv["unique_cluster_id_cov"]):
    print(f"Target: {cell_target}, Covariate: {cell_covariate}")
    row = neuron_only[(neuron_only["unique_cluster_id"] == cell_target) & (neuron_only["unique_cluster_id_cov"] == cell_covariate)]
    mouse = row["mouse"].values[0]
    day = row["day"].values[0]
    cluster_id = row["cluster_id"].values[0]
    cluster_id_cov = row["cluster_id_cov"].values[0]

    # Load data
    vr_folder = f'{source_path}M{mouse}/D{day:02}/VR/'
    spikes_path = vr_folder + f"sub-{mouse}_day-{day:02}_ses-VR_srt-kilosort4_clusters.npz"
    beh_path = vr_folder + f"sub-{mouse}_day-{day:02}_ses-VR_beh.nwb"
    beh = nap.load_file(beh_path)
    clusters = nap.load_file(spikes_path)
    #print(f'there are this many clusters before curation {len(clusters)}')
    clusters = curate_clusters(clusters)

    # Compute CCH
    CCH = compute_CCH(clusters, [cluster_id, cluster_id_cov])

    # Compute baseline CCH
    bin_size_sec = 0.001
    std_sec = 0.005
    support = 3 * std_sec
    support_bins = int(support / bin_size_sec)
    x = np.arange(-support, support + bin_size_sec, bin_size_sec)
    gaussian = np.exp(-0.5 * (x / std_sec) ** 2)
    gaussian[x == 0] *= 0.6
    gaussian /= gaussian.sum()
    CCH_baseline = xr.apply_ufunc(
        lambda x: convolve(x, gaussian, mode="same"),
        CCH,
        input_core_dims=[["lag"]],
        output_core_dims=[["lag"]],
        vectorize=True,
        output_dtypes=[float],
    )
    CCH_baseline.name = "cch_baseline"

    # Classify
    fc = xr.apply_ufunc(
        classify_CCH,
        CCH,
        CCH_baseline,
        input_core_dims=[["lag"], ["lag"]],
        vectorize=True,
        kwargs={"lags": CCH.coords["lag"].values, "bin_size_sec": 0.001},
        output_dtypes=[bool],
    )
    fc = fc.to_series().reset_index()
    fc = fc.rename({"source": "from", "target": "to", 0: "connected"}, axis=1)

    import matplotlib.pyplot as plt

    connected = fc[fc["connected"]]
    lag_arrays = [
        CCH.sel(source=fr, target=to)
        for fr, to in zip(connected["from"], connected["to"])
    ]

    from scipy.ndimage import gaussian_filter1d

    for i in range(len(lag_arrays)):
        plt.bar(
            x=CCH.coords["lag"].values,
            height=gaussian_filter1d(lag_arrays[i].values, sigma=1.0),
            width=0.001,
        )
        plt.show()


In [ ]:
x

In [ ]:

if __name__ == "__main__":
    parser = generate_argument_parser(
        "Analysing functional connectivity for Nolan lab ephys data.",
        script_type="experimental",
    )
    parser.add_argument(
        "--bin_size_sec",
        type=float,
        default=0.001,
        help="Bin size in seconds for decoding.",
    )
    args = parser.parse_args()

    # Get head direction cells
    hd_OF1 = pd.read_parquet(
        args.storage
        / "sessions"
        / f"M{args.mouse}"
        / f"D{args.day:0>2}"
        / "OF1"
        / "tuning_scores"
        # / "hd_mean_vector_length.parquet"
        / "hd_information.parquet"
    )
    stab_OF1 = pd.read_parquet(
        args.storage
        / "sessions"
        / f"M{args.mouse}"
        / f"D{args.day:0>2}"
        / "OF1"
        / "tuning_scores"
        / "hd_stability.parquet"
    )
    hd_OF2 = pd.read_parquet(
        args.storage
        / "sessions"
        / f"M{args.mouse}"
        / f"D{args.day:0>2}"
        / "OF2"
        / "tuning_scores"
        # / "hd_mean_vector_length.parquet"
        / "hd_information.parquet"
    )
    stab_OF2 = pd.read_parquet(
        args.storage
        / "sessions"
        / f"M{args.mouse}"
        / f"D{args.day:0>2}"
        / "OF2"
        / "tuning_scores"
        / "hd_stability.parquet"
    )
    hd_cells = hd_OF1[
        hd_OF1["sig"]
        & hd_OF2["sig"]
        & stab_OF1["sig"]
        & stab_OF2["sig"]
        & (hd_OF1["brain_region"].str.contains("ENT"))
    ]["cluster_id"].values
    print("Num hd cells: ", len(hd_cells))

    if len(hd_cells) < 1:
        raise ValueError(
            f"Not a single head direction cell for mouse {args.mouse} day {args.day}: {len(hd_cells)}"
        )

    # Get grid cells
    grid_OF1 = pd.read_parquet(
        args.storage
        / "sessions"
        / f"M{args.mouse}"
        / f"D{args.day:0>2}"
        / "OF1"
        / "tuning_scores"
        / "grid_score.parquet"
    )
    grid_OF2 = pd.read_parquet(
        args.storage
        / "sessions"
        / f"M{args.mouse}"
        / f"D{args.day:0>2}"
        / "OF2"
        / "tuning_scores"
        / "grid_score.parquet"
    )
    grid_cells = hd_OF1[grid_OF1["sig"] & (hd_OF1["brain_region"].str.contains("ENT"))][
        "cluster_id"
    ].values

    if len(grid_cells) < 1:
        raise ValueError(
            f"Not a single grid cell for mouse {args.mouse} day {args.day}: {len(hd_cells)}"
        )
    print("Num grid cells: ", len(grid_cells))

    # Load sessions
    data = {}
    for session_type in ["OF1", "OF2", "VR"]:
        args.session_type = session_type
        session, session_path, clusters = load_session(args)
        data[session_type] = {
            "clusters": clusters,
            "moving": session["moving"],
        }

    # Compute CCH
    CCH = compute_CCH(data["OF1"]["clusters"], list(set(hd_cells) | set(grid_cells)))

    # Compute baseline CCH
    std_sec = 0.005
    support = 3 * std_sec
    support_bins = int(support / args.bin_size_sec)
    x = np.arange(-support, support + args.bin_size_sec, args.bin_size_sec)
    gaussian = np.exp(-0.5 * (x / std_sec) ** 2)
    gaussian[x == 0] *= 0.6
    gaussian /= gaussian.sum()
    CCH_baseline = xr.apply_ufunc(
        lambda x: convolve(x, gaussian, mode="same"),
        CCH,
        input_core_dims=[["lag"]],
        output_core_dims=[["lag"]],
        vectorize=True,
        output_dtypes=[float],
    )
    CCH_baseline.name = "cch_baseline"

    # Classify
    fc = xr.apply_ufunc(
        classify_CCH,
        CCH,
        CCH_baseline,
        input_core_dims=[["lag"], ["lag"]],
        vectorize=True,
        kwargs={"lags": CCH.coords["lag"].values, "bin_size_sec": 0.001},
        output_dtypes=[bool],
    )
    fc = fc.to_series().reset_index()
    fc = fc.rename({"source": "from", "target": "to", 0: "connected"}, axis=1)

    import matplotlib.pyplot as plt

    connected = fc[fc["connected"]]
    lag_arrays = [
        CCH.sel(source=fr, target=to)
        for fr, to in zip(connected["from"], connected["to"])
    ]

    from scipy.ndimage import gaussian_filter1d

    for i in range(len(lag_arrays)):
        plt.bar(
            x=CCH.coords["lag"].values,
            height=gaussian_filter1d(lag_arrays[i].values, sigma=1.0),
            width=0.001,
        )
        plt.show()
    quit()

    fc["from_type"] = fc["from"].apply(
        lambda x: "HG"
        if x in hd_cells and x in grid_cells
        else "H"
        if x in hd_cells
        else "G"
        if x in grid_cells
        else "/"
    )
    fc = pd.merge(
        fc,
        hd_OF1[["cluster_id", "preferred"]].rename(
            {"preferred": "from_preferred"}, axis=1
        ),
        left_on="from",
        right_on="cluster_id",
        how="inner",
    ).drop("cluster_id", axis=1)
    fc["to_type"] = fc["to"].apply(
        lambda x: "HG"
        if x in hd_cells and x in grid_cells
        else "H"
        if x in hd_cells
        else "G"
        if x in grid_cells
        else "/"
    )
    fc = pd.merge(
        fc,
        hd_OF1[["cluster_id", "preferred"]].rename(
            {"preferred": "to_preferred"}, axis=1
        ),
        left_on="to",
        right_on="cluster_id",
        how="inner",
    ).drop("cluster_id", axis=1)
    print(fc)
    fc.to_parquet(session_path.parent / "hd_grid_connections.parquet")
    print(f"{fc['connected'].sum()}, {fc['connected'].sum() / len(fc):.2f}% connected")
    # plt.plot(ccg.index.values, ccg.values[:, t.values].mean(axis=1), label="raw")
    # plt.plot(
    #    ccg.index.values,
    #    baseline_ccg.values[:, t.values].mean(axis=1),
    #    label="baseline",
    # )
    # plt.axvspan(0.0007, 0.0047, facecolor="green", alpha=0.3, edgecolor="none")
    # plt.legend()
    # plt.show()
